# Cardiovascular disease: exploratory analysis

Recorded disease classification, not future-event prediction. No TG measurement is available. See README for provenance limitations.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))
from src.data import load_data, quality_flags
df = load_data()
df.shape

## Structure, missingness and descriptive statistics
Age is in days; ordinal codes are not laboratory concentrations.

In [ ]:
display(df.dtypes.to_frame("dtype"))
display(df.isna().sum().to_frame("missing"))
display(df.describe().T.assign(median=df.median(), variance=df.var()))
display(df.cardio.value_counts().to_frame("count"))

## Data-quality review
Flags prompt investigation. They do not automatically justify deleting rows. Inspect blood-pressure extremes and document any later exclusion policy.

In [ ]:
display(quality_flags(df).sum().to_frame("flagged_rows"))
display(df[["ap_hi", "ap_lo", "height", "weight"]].quantile([0, .01, .5, .99, 1]))
print("Duplicate records excluding ID:", df.drop(columns="id").duplicated().sum())

## Distributions and outcome relationships
Plot clipping below is for readability only; the underlying data is unchanged.

In [ ]:
view = df.assign(age_years=df.age / 365.25)
view[["age_years", "height", "weight"]].hist(bins=35, figsize=(12, 3))
plt.tight_layout()
plt.show()
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, col in zip(axes, ["age_years", "weight", "ap_hi"]):
    sns.boxplot(data=view, x="cardio", y=col, ax=ax, showfliers=False)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df.drop(columns="id").corr(method="spearman"), center=0, cmap="coolwarm", ax=ax)
ax.set_title("Spearman associations: not causal effects")
plt.tight_layout()
plt.show()

## Interpretation to complete
Describe the actual plots, explain quality decisions and connect findings to research questions. Do not infer population prevalence from this selected sample. Run `python -m src.train` from the root for the reproducible baseline. Hyperparameter tuning and feature explanations remain future work.